In [31]:
%pip install transformers==4.46.0

Note: you may need to restart the kernel to use updated packages.


In [32]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

In [33]:
import os
from pathlib import Path

In [34]:
import os
from pathlib import Path

REPO_ROOT = Path.home() / "market-sentiment-analysis-11"
os.chdir(REPO_ROOT)

import pandas as pd
df = pd.read_parquet("data/processed/gdelt_ohlcv_join.parquet")
print(f"loaded {len(df):,} rows")
print(df.columns.tolist())
print(f"{df['article_date'].min().date()} to {df['article_date'].max().date()}")
print(sorted(df['ticker'].unique()))

loaded 12,523 rows
['seendate', 'url', 'title', 'language', 'domain', 'socialimage', 'company', 'ticker', 'date', 'sentiment_score', 'sentiment_confidence', 'sentiment_label', 'article_date', 'price_date', 'next_open', 'next_high', 'next_low', 'next_close', 'next_adj_close', 'next_volume']
2024-02-08 to 2026-02-23
['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']


In [35]:
df.head(5)

,seendate,url,title,language,domain,socialimage,company,ticker,date,sentiment_score,sentiment_confidence,sentiment_label,article_date,price_date,next_open,next_high,next_low,next_close,next_adj_close,next_volume
0,2024-02-08 00:00:00+00:00,https://www.businesstimes.com.sg/companies-mar...,Arm soars after expansion into new markets buo...,English,businesstimes.com.sg,https://static1.businesstimes.com.sg/s3fs-publ...,Apple,AAPL,2024-02-08,0.815164,0.877368,positive,2024-02-08,2024-02-09,188.649994,189.990005,188.0,188.850006,187.146774,45155200
1,2024-02-08 00:45:00+00:00,https://www.hucknalldispatch.co.uk/lifestyle/f...,Dutch Barn Orchard Vodka achieves nationwide l...,English,hucknalldispatch.co.uk,https://www.hucknalldispatch.co.uk/webimg/b25l...,Apple,AAPL,2024-02-08,0.927444,0.938555,positive,2024-02-08,2024-02-09,188.649994,189.990005,188.0,188.850006,187.146774,45155200
2,2024-02-08 00:45:00+00:00,https://www.fool.com/earnings/call-transcripts...,PayPal ( PYPL ) Q4 2023 Earnings Call Transcript,English,fool.com,None,Apple,AAPL,2024-02-08,-0.010950,0.934771,neutral,2024-02-08,2024-02-09,188.649994,189.990005,188.0,188.850006,187.146774,45155200
3,2024-02-08 01:15:00+00:00,https://www.nbcnewyork.com/news/business/money...,"Jim Cramer says recent moves in Apple , Chipot...",English,nbcnewyork.com,https://media.nbcnewyork.com/2023/11/107113454...,Apple,AAPL,2024-02-08,-0.320946,0.543592,neutral,2024-02-08,2024-02-09,188.649994,189.990005,188.0,188.850006,187.146774,45155200
4,2024-02-08 01:15:00+00:00,https://invezz.com/news/2024/02/07/disney-q1-e...,Disney Q1 earnings : dividend increased as DTC...,English,invezz.com,https://invezz.com/wp-content/uploads/2022/11/...,Apple,AAPL,2024-02-08,0.838071,0.908904,positive,2024-02-08,2024-02-09,188.649994,189.990005,188.0,188.850006,187.146774,45155200


In [36]:
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<bound method PreTrainedModel.eval of BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (La

In [37]:
titles = df["title"].fillna("").astype(str).tolist()
score = []
batch_size = 32

In [38]:
for i in tqdm(range(0, len(titles), batch_size)):
    batch = titles[i:i + batch_size]
    inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt")
    inputs = {k: v.to(device) for k,v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=1).cpu().numpy()
    
    for prob in probs:
        p_pos, p_neg, p_neu = prob
        score.append(float(p_pos - p_neg))


100%|██████████| 392/392 [02:12<00:00,  2.95it/s]


In [39]:
print(f"score list length: {len(score)}")
print(f"df length: {len(df)}")
print(f"First 5 FinBERT scores: {score[:5]}")

score list length: 12523
df length: 12523
First 5 FinBERT scores: [0.8151633143424988, 0.927444338798523, -0.010950125753879547, -0.32094883918762207, 0.8380698561668396]


In [40]:
df["sentiment_score"] = score
df["sentiment_present"] = [abs(s) > 0.1 for s in score]

print(f"Score range: [{df['sentiment_score'].min():.4f}, {df['sentiment_score'].max():.4f}]")
print(f"Score mean: {df['sentiment_score'].mean():.4f}")
print(f"Unique values: {df['sentiment_score'].nunique()}")
print(f"Sentiment present: {df['sentiment_present'].sum():,} / {len(df):,}")
df[["title", "sentiment_score", "sentiment_present"]].sample(5, random_state=42)

Score range: [-0.9679, 0.9479]
Score mean: 0.0235
Unique values: 12093
Sentiment present: 7,751 / 12,523


,title,sentiment_score,sentiment_present
11566,FinancialContent - The AI Inference Engine :...,0.915685,True
1328,Alphabet Inc . ( NASDAQ : GOOGL ) Shares Acqui...,-0.019151,False
6839,FinancialContent - The Barbell Bet : Balancing...,0.202174,True
11963,Nasdaq to lead Wall Street higher as sector ro...,0.917776,True
9529,"How To Trade SPY , QQQ , AAPL , MSFT , NVDA , ...",0.029718,False


In [41]:
print([c for c in df.columns  if 'sentiment' in c.lower()])

['sentiment_score', 'sentiment_confidence', 'sentiment_label', 'sentiment_present']


In [43]:
print(df["sentiment_score"].value_counts())
print(f"\nSentiment present: {df['sentiment_present'].sum():,} / {len(df):,} ({100*df['sentiment_present'].mean():.1f}%)")
print(f"Score range: [{df['sentiment_score'].min():.3f}, {df['sentiment_score'].max():.3f}]")
print(f"Score mean: {df['sentiment_score'].mean():.3f}")
df[["title", 'sentiment_score', 'sentiment_confidence', 'sentiment_label', 'sentiment_present']].head(10)


sentiment_score
 0.424002    7
 0.268636    7
 0.140254    6
-0.942602    5
-0.008322    5
            ..
 0.912808    1
 0.100603    1
 0.054269    1
 0.026473    1
 0.115710    1
Name: count, Length: 12093, dtype: int64

Sentiment present: 7,751 / 12,523 (61.9%)
Score range: [-0.968, 0.948]
Score mean: 0.023


,title,sentiment_score,sentiment_confidence,sentiment_label,sentiment_present
0,Arm soars after expansion into new markets buo...,0.815163,0.877368,positive,True
1,Dutch Barn Orchard Vodka achieves nationwide l...,0.927444,0.938555,positive,True
2,PayPal ( PYPL ) Q4 2023 Earnings Call Transcript,-0.010950,0.934771,neutral,False
3,"Jim Cramer says recent moves in Apple , Chipot...",-0.320949,0.543592,neutral,True
4,Disney Q1 earnings : dividend increased as DTC...,0.838070,0.908904,positive,True
5,"Cramer : Moves in Apple , Chipotle and Eli Lil...",-0.134405,0.770233,neutral,True
6,Perfect storm : B . C . fruit farmers warn o...,-0.933047,0.951578,negative,True
7,Why investors should be more excited about the...,0.032378,0.911672,neutral,False
8,"Apple , Chipotle And Eli Lilly Stocks Baffle J...",0.003489,0.907371,neutral,False
9,Apple wins lawsuit over compensation of CEO Ti...,-0.904503,0.923187,negative,True


In [44]:
df.to_parquet("data/processed/gdelt_ohlcv_join_finbert.parquet")
print("saved!")

saved!
